In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import pandas as pd

from dotenv import load_dotenv
from src import scoring, prompts, config, evaluate

In [ ]:
def ew_rejudge(
    judge_instruction_name: str = "ji2",
    source_csv_path: str | Path = "/root/work/p25-plexam/experiments/model_study.csv",
    output_csv_path: str | Path | None = None,
    models=None,
    judge_models=None,
    rejudge_already_scored_models: bool = True,
    generation_columns=None,
    task_index_column: str = "index",
    gpbam_path: str | Path = "data/gpbam.json",
    require_confirmation: bool = True,
    verbose: bool = True,
):
    """
    Re-judge an already-generated set of answers under a (possibly different) judge
    instruction, without regenerating the answers. Notebook version of
    essay_writing_rejudge.py.

    Every parameter mirrors a constant that was hardcoded in the script. Pass
    `require_confirmation=False` to skip the interactive 'yes' gate when iterating.

    Args:
        judge_instruction_name: Key into `prompts.JUDGE_INSTRUCTIONS_BY_NAME` selecting
            the judge instruction actually sent to the judge (e.g. "ji1", "ji2"). Also
            names the default `output_csv_path` folder/file.
            
        source_csv_path: Read-only CSV of answers generated in an earlier run — the
            answers to re-judge (no regeneration happens here). Must contain 'model' and
            'answer' columns; optionally the generation-metadata columns ('message',
            'qa_prompt', 'prompt_tokens', 'completion_tokens', 'total_tokens', 'total_cost',
            'time', 'temperature', 'top_p', 'finish_reason') and a task-index column, which
            are carried over unchanged. Only the answers (plus that metadata) are used: any
            judge scores already in this file are ignored, not read or copied, since fresh
            scores are computed. This file is never modified or written to.

        output_csv_path: Where re-judged results are written. If None, defaults to a
            path derived from `judge_instruction_name`. Also acts as a resume cache:
            existing rows are loaded and re-saved after every model.
        models: Iterable of model names whose answers to re-judge. Defaults to
            `config.MODELS_DEV`. Models absent from `source_csv_path` are skipped.
        judge_models: Model names forming the judge ensemble. Defaults to
            `config.JUDGE_MODELS_DEV`.
        rejudge_already_scored_models: Controls what happens to a model already
            present in `output_csv_path` (i.e. re-judged in a prior run) — not the
            source answers. True re-judges it and replaces its old result rows; False
            skips it, so a re-run resumes an interrupted run / adds new models without
            re-scoring (and re-paying for) ones already done.
        generation_columns: Generation metadata columns (tokens, cost, time, etc.)
            carried over unchanged from source to output. Defaults to a standard set;
            columns missing from the source are ignored.
        task_index_column: Column giving each answer's gpbam task index, used to align
            answer[i] with facts[i]/solutions[i]. Falls back to CSV row order if absent.
            Completeness (every gpbam task covered exactly once) is enforced regardless.
        gpbam_path: Path to the gpbam JSON providing the `facts` and `solutions` each
            answer is judged against.
        require_confirmation: If True, print a path/instruction summary and gate the run
            behind an interactive 'yes' prompt. Set False to run non-interactively.
        verbose: If True, print per-model / per-judge progress while re-judging.

    Returns:
        The re-judged results DataFrame (also written to `output_csv_path`).
    """

    load_dotenv(override=True)

    # --- defaults that depend on other args / modules ---
    if models is None:
        models = config.MODELS_DEV
    if judge_models is None:
        judge_models = config.JUDGE_MODELS_DEV
    if generation_columns is None:
        # Columns describing generation (not judging) to carry over unchanged.
        generation_columns = ['message', 'qa_prompt', 'prompt_tokens', 'completion_tokens',
                              'total_tokens', 'total_cost', 'time', 'temperature', 'top_p', 'finish_reason']

    judge_instruction = prompts.JUDGE_INSTRUCTIONS_BY_NAME[judge_instruction_name]

    source_csv_path = Path(source_csv_path)
    if output_csv_path is None:
        output_csv_path = (
            f"/root/work/p25-plexam/experiments/zubaers_result/re_judging/essay_writing/"
            f"{judge_instruction_name}/{judge_instruction_name}_result_rejudged.csv"
        )
    output_csv_path = Path(output_csv_path)

    gpbam_df = pd.read_json(gpbam_path)

    # --- safety checkpoint: confirm paths and instruction before running ---
    print("\n" + "=" * 80)
    print("RE-JUDGE PATH CHECK")
    print("=" * 80)
    print(f"Source answers CSV (not regenerated):\n{source_csv_path}")
    print(f"Output results CSV:\n{output_csv_path}")
    print("=" * 80)
    print(f"Judge instruction ({judge_instruction_name})")
    print(judge_instruction[:])
    print("=" * 80)
    print(f"Allow overwrite existing model rows: {rejudge_already_scored_models}")
    print("\nBefore continuing, verify:")
    print("1. The source CSV holds the answers you intend to re-judge (no regeneration happens here).")
    print("2. The output folder/filename identify the judge instruction | rag | no-rag | being applied.")
    print("3. The instruction text printed above matches judge_instruction_name "
          "(this is what's actually sent.")
    print("=" * 80)

    if require_confirmation:
        confirmation = input("Type \"yes\" to continue running the script: ").strip().lower()
        if confirmation != "yes":
            raise RuntimeError(
                "Execution stopped. Please set the paths/instruction correctly before running again."
            )

    # --- load source answers + any existing re-judged results ---
    if not source_csv_path.exists():
        raise FileNotFoundError(f"Source CSV not found: {source_csv_path}\n You must first generate answers for the models you want to re-judge.")

    source_df = pd.read_csv(source_csv_path)
    assert 'model' in source_df and 'answer' in source_df

    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    if output_csv_path.exists():
        print(f"Loading existing re-judged results from {output_csv_path}")
        model_study = pd.read_csv(output_csv_path)
    else:
        print("No existing re-judged results found. Starting a fresh results DataFrame.")
        model_study = pd.DataFrame()

    completed_models = set(model_study["model"].dropna().unique()) if "model" in model_study else set()

    # --- judge ensemble ---
    judges = [scoring.Judge(model=model, prompt=prompts.build_judge_user(judge_instruction))
              for model in judge_models]
    judge = scoring.JudgeEnsemble(judges, verbose=True)

    # --- re-judge each model ---
    for model in dict.fromkeys(models):
        if model not in source_df["model"].unique():
            print(f"Skipping {model}: no existing generated answers for it in {source_csv_path}.")
            continue

        if model in completed_models and not rejudge_already_scored_models:
            print(f"Skipping {model}: already re-judged.")
            continue

        if model in completed_models:
            print(f"Overwriting {model}: removing existing rows before rerun.")
            model_study = model_study[model_study["model"] != model].copy()

        print(f"Re-judging {model}")

        model_answers = source_df[source_df["model"] == model]

        # --- Completeness + alignment guard ----------------------------------
        # rejudge pairs answer[i] with facts[i]/solutions[i], so the answers must
        # (1) cover every gpbam task exactly once, and (2) be in gpbam task order.
        if task_index_column in model_answers.columns:
            task_idx = model_answers[task_index_column].tolist()   # explicit, verifiable
            order_verified = True
        else:
            task_idx = list(range(len(model_answers)))             # fall back to CSV order
            order_verified = False

        # Completeness is ALWAYS enforced (works with or without an index column):
        if sorted(task_idx) != list(range(len(gpbam_df))):
            raise ValueError(
                f"{model}: source answers do not cover gpbam's {len(gpbam_df)} tasks "
                f"exactly once (got {len(task_idx)} rows). Refusing to re-judge to "
                f"avoid partial/misaligned scores. Fix {source_csv_path}."
            )

        if not order_verified:
            print(f"  Note: no '{task_index_column}' column for {model}; trusting CSV "
                  f"row order as gpbam task order (do not manually reorder this CSV).")

        # Reorder so answer[i] lines up with facts[i] (a no-op when task_idx is 0..N-1).
        model_answers = (model_answers.assign(_task_idx=task_idx)
                         .sort_values("_task_idx").drop(columns="_task_idx")
                         .reset_index(drop=True))
        answers = model_answers["answer"].values

        gen_cols = [c for c in generation_columns if c in model_answers.columns]
        generation_info = model_answers[gen_cols]

        model_df = evaluate.rejudge_model(
            answers,
            tasks=gpbam_df.facts.values,          # full set — no [:N] slice
            solutions=gpbam_df.solutions.values,
            judge=judge,
            generation_info=generation_info,
            model_name=model,
            verbose=verbose,
        )

        model_study = pd.concat([model_study, model_df], ignore_index=True)

        # Save after every model so interrupted runs can resume safely.
        model_study.to_csv(output_csv_path, index=False)
        completed_models.add(model)

    return model_study


In [ ]:
'''
# Reproduces the original script exactly (still prompts for 'yes'):
df = ew_rejudge()

# Notebook-friendly: skip the input() prompt
df = ew_rejudge(require_confirmation=True)

# Re-judge the lawrag answers under a different instruction, to a chosen output:
df = ew_rejudge(
    judge_instruction_name="ji2",
    source_csv_path=".../with_rag/ew_model_study_with_rag_generation_only.csv",
    output_csv_path=".../with_rag/ji2/with_rag_ji2_result.csv",
    require_confirmation=True,
)
'''

In [ ]:
rj = ew_rejudge(
    judge_instruction_name="ji2",                      # same instruction as the file
    source_csv_path="/root/work/p25-plexam/experiments/zubaers_result/essay_writing/with_rag/ji2/with_rag_ji2_result.csv",
    output_csv_path="/root/work/p25-plexam/experiments/zubaers_result/re_judging/essay_writing/with_rag_ji2/deepseek_v32_qwen.csv",
    models=["deepseek/deepseek-v3.2"],                 # only this model
    judge_models=["qwen/qwen3.6-35b-a3b"],             # only the missing judge
    # generation_columns=["index"],                      # carry index so we can merge cleanly
    require_confirmation=False,
)


In [ ]:
import shutil, pandas as pd

MAIN  = "/root/work/p25-plexam/experiments/zubaers_result/essay_writing/with_rag/ji2/with_rag_ji2_result.csv"
MODEL = "deepseek/deepseek-v3.2"
shutil.copy(MAIN, MAIN + ".bak")

main = pd.read_csv(MAIN)
rj   = pd.read_csv("/root/work/p25-plexam/experiments/zubaers_result/re_judging/essay_writing/with_rag_ji2/deepseek_v32_qwen.csv")

qwen_cols = [c for c in rj.columns if c.endswith("(qwen3.6-35b-a3b)")]   # plain, NOT ...FP8
assert len(qwen_cols) == 4, qwen_cols                                     # score_/judgement_/_text_/_finish_reason_

rj_by_idx = rj.set_index("index")[qwen_cols]
rows = main.loc[main["model"] == MODEL].sort_values("index")
assert list(rows["index"]) == list(rj_by_idx.index), "index mismatch — do not merge"

for c in qwen_cols:
    main.loc[rows.index, c] = rj_by_idx.loc[rows["index"], c].values

main.to_csv(MAIN, index=False)


In [ ]:
pd.read_csv("/root/work/p25-plexam/experiments/zubaers_result/re_judging/essay_writing/with_rag_ji2/deepseek_v32_qwen.csv")